In [1]:
!pip install -Uqq ucimlrepo
!pip install -Uqq fastbook

In [2]:
from fastbook import *
import fastbook
fastbook.setup_book

<function fastbook.setup_book()>

In [3]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
real_estate_valuation = fetch_ucirepo(id=477) 
  
# data (as pandas dataframes) 
X = real_estate_valuation.data.features 
y = real_estate_valuation.data.targets 
  
# metadata 
#print(f"Metadata: ({real_estate_valuation.metadata})\n\n") 
  
# variable information 
#print(f"Variables: ({real_estate_valuation.variables})\n\n") 
print(f"Columns: {X.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 6 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   X1 transaction date                     414 non-null    float64
 1   X2 house age                            414 non-null    float64
 2   X3 distance to the nearest MRT station  414 non-null    float64
 3   X4 number of convenience stores         414 non-null    int64  
 4   X5 latitude                             414 non-null    float64
 5   X6 longitude                            414 non-null    float64
dtypes: float64(5), int64(1)
memory usage: 19.5 KB
Columns: None


In [4]:
from fastai.data.transforms import RandomSplitter
trn_split,val_split=RandomSplitter(seed=42)(X)

In [5]:
trn_indep, val_indep = X.iloc[trn_split], X.iloc[val_split]
trn_dep, val_dep = y.iloc[trn_split], y.iloc[val_split]
trn_indep.shape, trn_dep.shape, val_indep.shape, val_dep.shape

((332, 6), (332, 1), (82, 6), (82, 1))

In [6]:
trn_indep.head()

,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude
290,2013.083,37.7,490.3446,0,24.97217,121.53471
383,2012.667,29.1,506.1144,4,24.97845,121.53889
226,2013.000,16.5,4082.0150,0,24.94155,121.50381
29,2013.083,7.1,451.2438,5,24.97563,121.54694
59,2013.083,13.3,336.0532,5,24.95776,121.53438


In [7]:
from fastai.imports import *

# Step 1: Create time features and normalize all independent variables

In [8]:
from sklearn import preprocessing
import numpy as np

def preprocess_independent_variables(indpt_vs, scaler = None):
    time_df = pd.DataFrame(index=indpt_vs.index)
    
    fractional, integral = np.modf(indpt_vs["X1 transaction date"])
    time_df["year"] = pd.Series(integral.astype(int))
    time_df["month"] = pd.Series(np.round(fractional*12))

    columns = ['X2 house age', 'X3 distance to the nearest MRT station', 'X4 number of convenience stores', 'X5 latitude', 'X6 longitude']
    indpt_vs_columns_df = pd.concat([time_df, indpt_vs[columns]], axis=1)
    
    min_max_scaler = scaler
    if min_max_scaler is None:
        s = preprocessing.MinMaxScaler()
        min_max_scaler = s.fit(indpt_vs_columns_df)
    indpt_vs_columns_minmax = min_max_scaler.transform(indpt_vs_columns_df)

    columns = ['year_norm', 'month_norm', "house_age",'distance_to_nearest_MRT_station', 'number_of_convenience_stores', 'latitude', 'longitude']
    df = pd.DataFrame(
        indpt_vs_columns_minmax,
        columns=columns,
        index=indpt_vs.index
    )
    return df, min_max_scaler

In [9]:
trn_indep, scaler = preprocess_independent_variables(trn_indep)
trn_indep.head()

,year_norm,month_norm,house_age,distance_to_nearest_MRT_station,number_of_convenience_stores,latitude,longitude
290,1.0,0.090909,0.860731,0.072233,0.0,0.485943,0.659694
383,0.0,0.727273,0.664384,0.074673,0.4,0.562046,0.704766
226,1.0,0.000000,0.376712,0.627820,0.0,0.114881,0.326504
29,1.0,0.090909,0.162100,0.066185,0.5,0.527872,0.791568
59,1.0,0.090909,0.303653,0.048366,0.5,0.311318,0.656135


# Step 2: Turn X and Y into tensors

In [10]:
idpnt_column_names = ['year_norm', 'month_norm', 'house_age', 'distance_to_nearest_MRT_station',
       'number_of_convenience_stores', 'latitude', 'longitude']
n_columns = len(idpnt_column_names)
train_indpt = tensor(trn_indep[idpnt_column_names])
n_columns, train_indpt[:5]

(7,
 tensor([[1.0000, 0.0909, 0.8607, 0.0722, 0.0000, 0.4859, 0.6597],
         [0.0000, 0.7273, 0.6644, 0.0747, 0.4000, 0.5620, 0.7048],
         [1.0000, 0.0000, 0.3767, 0.6278, 0.0000, 0.1149, 0.3265],
         [1.0000, 0.0909, 0.1621, 0.0662, 0.5000, 0.5279, 0.7916],
         [1.0000, 0.0909, 0.3037, 0.0484, 0.5000, 0.3113, 0.6561]]))

In [11]:
train_dpt = tensor(trn_dep)
train_dpt[:5]

tensor([[37.0000],
        [40.3000],
        [12.8000],
        [57.1000],
        [42.4000]])

# Step 3: Train linear model

In [12]:
def init_coeffs():
    coeffs = torch.rand(n_columns+1,1) - 0.5
    coeffs[-1] = 0.0
    return coeffs.requires_grad_()

In [13]:
def calc_preds(coeffs, train_indpt):
    weights = coeffs[:-1]
    intercept = coeffs[-1]
    return train_indpt @ weights + intercept

In [14]:
def calc_loss(coeffs, train_indpt, train_dep):
    preds = calc_preds(coeffs, train_indpt)
    return torch.abs(preds-train_dep).mean()
    

In [15]:
def update_coeffs(coeffs, lr=0.01):
    coeffs.sub_(coeffs.grad * lr)
    coeffs.grad.zero_()

In [16]:
def train_epoch(coeffs, lr=0.01):
    loss = calc_loss(coeffs, train_indpt, train_dpt)
    loss.backward()
    with torch.no_grad(): update_coeffs(coeffs, lr)
    print(f"{loss:.3}")

In [17]:
def train_model(epochs=30, lr=0.01):
    torch.manual_seed(442)
    coeffs = init_coeffs()
    for i in range(epochs): train_epoch(coeffs, lr)
    return coeffs

In [18]:
coeffs = train_model(epochs=100,lr=5.5)

38.3
23.4
11.7
9.45
9.2
8.99
8.79
8.59
8.41
8.26
8.1
7.95
7.8
7.65
7.51
7.38
7.24
7.12
7.02
6.93
6.86
6.81
6.76
6.72
6.69
6.65
6.62
6.59
6.57
6.55
6.5
6.48
6.46
6.46
6.42
6.41
6.39
6.36
6.35
6.34
6.36
6.34
6.33
6.31
6.33
6.3
6.31
6.28
6.3
6.26
6.27
6.26
6.29
6.29
6.31
6.33
6.3
6.32
6.28
6.3
6.24
6.26
6.22
6.22
6.22
6.25
6.21
6.21
6.2
6.23
6.2
6.24
6.2
6.24
6.2
6.25
6.2
6.24
6.19
6.24
6.21
6.24
6.19
6.2
6.16
6.16
6.15
6.15
6.14
6.14
6.14
6.13
6.13
6.13
6.13
6.13
6.12
6.12
6.12
6.12


In [19]:
def measure_loss(coeffs): 
    ivs, _ = preprocess_independent_variables(val_indep, scaler=scaler)
    return calc_loss(coeffs, tensor(ivs), tensor(val_dep))

loss = measure_loss(coeffs)
loss

tensor(6.6884, grad_fn=<MeanBackward0>)

In [20]:
def show_coeffs(): return dict(zip(idpnt_column_names, coeffs))
show_coeffs()

{'year_norm': tensor([3.1817], grad_fn=<UnbindBackward0>),
 'month_norm': tensor([4.7199], grad_fn=<UnbindBackward0>),
 'house_age': tensor([-12.4047], grad_fn=<UnbindBackward0>),
 'distance_to_nearest_MRT_station': tensor([-11.3187], grad_fn=<UnbindBackward0>),
 'number_of_convenience_stores': tensor([13.7021], grad_fn=<UnbindBackward0>),
 'latitude': tensor([18.6570], grad_fn=<UnbindBackward0>),
 'longitude': tensor([14.7824], grad_fn=<UnbindBackward0>)}

In [21]:
val_ivs, _ = preprocess_independent_variables(val_indep)

test_item = 0
pred = calc_preds(coeffs, tensor(val_ivs)[test_item])
val_indep.iloc[test_item], val_dep.iloc[test_item],pred

(X1 transaction date                       2012.75000
 X2 house age                                16.10000
 X3 distance to the nearest MRT station    4066.58700
 X4 number of convenience stores              0.00000
 X5 latitude                                 24.94297
 X6 longitude                               121.50342
 Name: 384, dtype: float64,
 Y house price of unit area    12.9
 Name: 384, dtype: float64,
 tensor([7.5503], grad_fn=<AddBackward0>))

# Step 4: Train neural net

In [22]:
def init_coeffs(n_hidden = 10):
    layer1 = (torch.rand(n_columns, n_hidden)-0.5).requires_grad_()
    layer2 = (torch.rand(n_hidden, 1)-.3).requires_grad_()
    const = torch.rand(1)[0].requires_grad_()
    return layer1, layer2, const

In [23]:
import torch.nn.functional as F

def calc_preds(coeffs, indpts):
    l1, l2, c = coeffs
    r = F.relu(indpts@l1)
    return r@l2 + c

In [24]:
def update_coeffs(coeffs, lr):
    for c in coeffs:
        c.sub_(c.grad * lr)
        c.grad.zero_()

In [25]:
coeffs = train_model(epochs=100,lr=.2)

37.2
36.7
36.1
35.1
33.6
31.6
28.4
23.6
16.5
9.99
9.34
9.03
8.84
8.66
8.49
8.33
8.18
8.04
7.91
7.78
7.66
7.54
7.42
7.31
7.21
7.12
7.05
6.99
6.94
6.89
6.85
6.81
6.77
6.74
6.71
6.69
6.66
6.64
6.62
6.6
6.58
6.57
6.55
6.54
6.53
6.52
6.5
6.49
6.48
6.47
6.46
6.45
6.44
6.44
6.43
6.43
6.41
6.4
6.39
6.39
6.37
6.37
6.36
6.35
6.34
6.34
6.33
6.32
6.32
6.31
6.31
6.3
6.3
6.3
6.31
6.3
6.33
6.39
6.52
6.82
6.84
6.86
6.83
6.86
6.87
6.85
6.83
6.84
6.83
6.87
6.84
6.85
6.85
6.83
6.81
6.83
6.8
6.83
6.79
6.83


In [26]:
def measure_loss(coeffs): 
    ivs, _ = preprocess_independent_variables(val_indep, scaler=scaler)
    return calc_loss(coeffs, tensor(ivs), tensor(val_dep))

loss = measure_loss(coeffs)
loss

tensor(7.1681, grad_fn=<MeanBackward0>)